# Accent-dependent ASR degradation after speech enhancement — HF-login version

This notebook runs a controlled audit:

1. Load **L2-ARCTIC** and **CMU ARCTIC**.
2. Select the **same 100 ARCTIC prompt IDs** across 7 accent groups:
   - American / native control from CMU ARCTIC
   - Arabic, Hindi, Korean, Mandarin, Spanish, Vietnamese from L2-ARCTIC
3. Listen to the first 5 matched prompts across accents.
4. Run this pipeline:

\[
	ext{raw speech}
ightarrow 	ext{Whisper ASR}
\]

\[
	ext{raw speech}
ightarrow 	ext{add controlled noise}
ightarrow 	ext{Whisper ASR}
\]

\[
	ext{raw speech}
ightarrow 	ext{add controlled noise}
ightarrow 	ext{speech enhancement}
ightarrow 	ext{Whisper ASR}
\]

5. Calculate WER/CER, absolute degradation, relative degradation, helped/hurt rates, and plots.

## Loader modes

This version explicitly supports three L2-ARCTIC loading paths:

- `hf_gated`: use `KoelLabs/L2Arctic` after logging into Hugging Face and accepting dataset access terms.
- `hf_zip_mirror`: use a speaker-zip mirror such as `chikingsley/l2-arctic-release-v5.0`.
- `local_l2_root`: use a manually downloaded/extracted L2-ARCTIC folder.

Start with `SPEAKERS_PER_ACCENT = 1` and `N_PROMPTS = 10` as a smoke test before running the full setup.


## 0. Install dependencies

Run this once. If Colab asks you to restart the runtime after installing `speechbrain` or `torchaudio`, restart and continue from imports.


In [ ]:
%pip install -q datasets[audio] huggingface_hub transformers accelerate jiwer soundfile librosa scipy pandas matplotlib tqdm torchaudio speechbrain ipywidgets

## 1. Imports and configuration


In [ ]:
import os
import re
import json
import math
import random
import zipfile
import shutil
import warnings
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import soundfile as sf
import librosa
from scipy.signal import butter, lfilter

from datasets import load_dataset, Audio
from huggingface_hub import list_repo_files, hf_hub_download, snapshot_download, notebook_login, login, get_token
from transformers import pipeline
from jiwer import wer, cer, Compose, ToLowerCase, RemovePunctuation, RemoveMultipleSpaces, Strip

from IPython.display import Audio as IPAudio, display, Markdown

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 0 if torch.cuda.is_available() else -1
TORCH_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", TORCH_DEVICE)


In [ ]:
# -----------------------------
# Dataset configuration
# -----------------------------

# Preferred path if you have accepted the HF gated dataset terms.
# choices: "hf_gated", "hf_zip_mirror", "local_l2_root"
L2_SOURCE = "hf_gated"

# Authentication options for gated HF datasets.
# choices: "notebook_login", "colab_secret", "env", "none"
# - notebook_login: prompts you to paste a token securely.
# - colab_secret: reads a secret named HF_TOKEN from Colab Secrets.
# - env: reads os.environ["HF_TOKEN"].
# - none: use only public datasets/repos.
HF_AUTH_MODE = "notebook_login"
HF_TOKEN_SECRET_NAME = "HF_TOKEN"

L2_GATED_REPO = "KoelLabs/L2Arctic"
#L2_GATED_SPLIT = "train"
L2_GATED_SPLIT = "scripted"

# Fallback: speaker-level zip archive mirror on HF.
# Use this if the gated dataset loader is inconvenient or the schema changes.
L2_ZIP_REPO = "chikingsley/l2-arctic-release-v5.0"

# If you manually download/extract L2-ARCTIC, set this and use L2_SOURCE="local_l2_root".
# Expected structure: root/SPEAKER/wav/*.wav and root/SPEAKER/transcript/*.txt
LOCAL_L2_ROOT = Path("/content/l2arctic_release_v5.0")

# CMU ARCTIC HF mirror. Each split is a speaker code.
CMU_DATASET_ID = "MikhailT/cmu-arctic"
CMU_NATIVE_SPEAKERS = ["bdl", "slt", "rms", "clb"]  # US English speakers in CMU ARCTIC

# L2-ARCTIC speaker mapping. Edit this if your release uses different labels.
L2_SPEAKERS_BY_ACCENT = {
    "Arabic": ["ABA", "SKA", "YBAA", "ZHAA"],
    "Hindi": ["ASI", "RRBI", "SVBI", "TNI"],
    "Korean": ["HJK", "HKK", "YDCK", "YKWK"],
    "Mandarin": ["BWC", "LXC", "NCC", "TXHC"],
    "Spanish": ["EBVS", "ERMS", "MBMPS", "NJS"],
    "Vietnamese": ["HQTV", "PNV", "THV", "TLV"],
}

# Smoke-test values: SPEAKERS_PER_ACCENT=1, N_PROMPTS=10.
# Full initial experiment: SPEAKERS_PER_ACCENT=4, N_PROMPTS=100.
SPEAKERS_PER_ACCENT = 1
N_PROMPTS = 10
TARGET_SR = 16000

# Whisper model: base.en is good for initial debugging. small.en is stronger but slower.
WHISPER_MODEL = "openai/whisper-base.en"

# Controlled noise setting.
NOISE_TYPE = "babble"  # options: "white", "pink", "babble"
SNR_DB = 5

# Initial speech enhancement models.
# Keep this list short first; setup/runtime grows fast.
ENHANCERS_TO_RUN = ["sepformer_dns4", "metricgan_plus"]

# Debug cap after matched prompts are selected. None = run full selected set.
MAX_ROWS_DEBUG = None  # e.g., 100

OUT_DIR = Path("accent_enhancement_audit_outputs")
CACHE_DIR = OUT_DIR / "cache"
L2_EXTRACT_DIR = CACHE_DIR / "l2_arctic_extracted"
AUDIO_WORK_DIR = OUT_DIR / "audio_work"
for d in [OUT_DIR, CACHE_DIR, L2_EXTRACT_DIR, AUDIO_WORK_DIR]:
    d.mkdir(exist_ok=True, parents=True)

HF_TOKEN = os.environ.get("HF_TOKEN", None)

print("L2_SOURCE:", L2_SOURCE)
print("HF_AUTH_MODE:", HF_AUTH_MODE)
print("Output dir:", OUT_DIR.resolve())

## 1.1 Hugging Face authentication

Run this cell if `L2_SOURCE = "hf_gated"`.

Before running it:

1. Open the gated L2-ARCTIC dataset page on Hugging Face.
2. Accept the access terms.
3. Create a Hugging Face access token.
4. Run the cell below and paste the token when prompted.

If you use Colab Secrets, add a secret named `HF_TOKEN`, set `HF_AUTH_MODE = "colab_secret"`, and rerun this cell.


In [ ]:
def setup_hf_auth():
    """Authenticate to Hugging Face if needed and return an available token."""
    global HF_TOKEN

    if HF_AUTH_MODE == "notebook_login":
        print("Launching Hugging Face notebook login. Paste your token when prompted.")
        notebook_login()
        HF_TOKEN = get_token()

    elif HF_AUTH_MODE == "colab_secret":
        try:
            from google.colab import userdata
            HF_TOKEN = userdata.get(HF_TOKEN_SECRET_NAME)
            if HF_TOKEN is None:
                raise ValueError(f"Colab secret {HF_TOKEN_SECRET_NAME!r} was not found.")
            login(token=HF_TOKEN)
            print(f"Logged in using Colab secret {HF_TOKEN_SECRET_NAME!r}.")
        except Exception as e:
            raise RuntimeError(
                "Could not authenticate with Colab Secrets. Either add HF_TOKEN in Colab Secrets "
                "or set HF_AUTH_MODE='notebook_login'."
            ) from e

    elif HF_AUTH_MODE == "env":
        HF_TOKEN = os.environ.get("HF_TOKEN")
        if HF_TOKEN is None:
            raise ValueError("HF_AUTH_MODE='env' but os.environ['HF_TOKEN'] is not set.")
        login(token=HF_TOKEN)
        print("Logged in using HF_TOKEN from environment.")

    elif HF_AUTH_MODE == "none":
        HF_TOKEN = os.environ.get("HF_TOKEN") or get_token()
        if HF_TOKEN:
            print("Using an existing cached/env Hugging Face token.")
        else:
            print("No Hugging Face token configured. Public repos only.")

    else:
        raise ValueError(f"Unknown HF_AUTH_MODE: {HF_AUTH_MODE}")

    print("Token available:", HF_TOKEN is not None)

    if HF_TOKEN is not None:
      os.environ["HF_TOKEN"] = HF_TOKEN
      os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN


    return HF_TOKEN

# Only authenticate automatically when the gated L2 source is selected.
if L2_SOURCE == "hf_gated":
    setup_hf_auth()
else:
    HF_TOKEN = os.environ.get("HF_TOKEN") or get_token()
    print("Skipping explicit login because L2_SOURCE is not hf_gated. Token available:", HF_TOKEN is not None)

## 2. Utility functions


In [ ]:
UTT_RE = re.compile(r"(a\d{4})", re.IGNORECASE)

text_norm = Compose([
    ToLowerCase(),
    RemovePunctuation(),
    RemoveMultipleSpaces(),
    Strip(),
])

def extract_utt_id(x):
    # Extract ARCTIC utterance id like a0001 from a file name/path/string.
    s = str(x)
    m = UTT_RE.search(s)
    if not m:
        return None
    return m.group(1).lower()


def normalize_ref_text(x):
    if x is None:
        return ""
    x = str(x).strip()
    # L2 transcript files sometimes include filename/id tokens. Remove if present.
    parts = x.split()
    if parts and UTT_RE.search(parts[0]):
        x = re.sub(r"^[A-Za-z0-9_\-]+\s+", "", x)
    return x.strip()


def load_audio_any(audio_or_path, target_sr=16000):
    """
    Robust audio loader for:
    - Hugging Face old-style Audio dict: {"array": ..., "sampling_rate": ...}
    - Hugging Face new-style TorchCodec AudioDecoder
    - local file paths
    - raw numpy arrays
    """

    import numpy as np
    import librosa
    import soundfile as sf

    # Case 1: Hugging Face old-style decoded audio dict
    if isinstance(audio_or_path, dict):
        if "array" in audio_or_path and "sampling_rate" in audio_or_path:
            wav = np.asarray(audio_or_path["array"], dtype=np.float32)
            sr = int(audio_or_path["sampling_rate"])

        elif "path" in audio_or_path and audio_or_path["path"] is not None:
            wav, sr = sf.read(audio_or_path["path"])

        else:
            raise ValueError(f"Unsupported audio dict keys: {audio_or_path.keys()}")

    # Case 2: Hugging Face new TorchCodec AudioDecoder
    elif hasattr(audio_or_path, "get_all_samples"):
        samples = audio_or_path.get_all_samples()

        # TorchCodec usually returns samples.data as torch.Tensor [channels, time]
        data = samples.data
        if hasattr(data, "detach"):
            data = data.detach().cpu().numpy()

        wav = np.asarray(data, dtype=np.float32)
        sr = int(samples.sample_rate)

        # Convert [channels, time] -> [time] if needed
        if wav.ndim == 2:
            wav = wav.mean(axis=0)

    # Case 3: path-like object
    elif isinstance(audio_or_path, (str, Path)):
        wav, sr = sf.read(str(audio_or_path))

    # Case 4: already an array
    else:
        wav = np.asarray(audio_or_path, dtype=np.float32)
        sr = target_sr

    # Convert stereo/multichannel [time, channels] -> mono
    if wav.ndim > 1:
        if wav.shape[0] < wav.shape[1]:
            # likely [channels, time]
            wav = wav.mean(axis=0)
        else:
            # likely [time, channels]
            wav = wav.mean(axis=1)

    wav = wav.astype(np.float32)

    # Resample
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr).astype(np.float32)
        sr = target_sr

    # Avoid weird amplitude scaling issues
    if np.max(np.abs(wav)) > 1.5:
        wav = wav / np.max(np.abs(wav))

    return wav, sr


def write_wav(path, wav, sr=TARGET_SR):
    path = Path(path)
    path.parent.mkdir(exist_ok=True, parents=True)
    wav = np.asarray(wav, dtype=np.float32)
    wav = np.clip(wav, -1.0, 1.0)
    sf.write(str(path), wav, sr)
    return path


def safe_id(s):
    return re.sub(r"[^A-Za-z0-9_\-]+", "_", str(s))


def rms(x):
    x = np.asarray(x, dtype=np.float32)
    return float(np.sqrt(np.mean(x**2) + 1e-12))


def active_len_sec(wav, sr=TARGET_SR):
    return len(wav) / sr


## 3. Load CMU ARCTIC native/American control

This uses 4 CMU ARCTIC native US English speakers: `bdl`, `slt`, `rms`, `clb`.


In [ ]:
def normalize_prompt_text(text):
    import re
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def load_cmu_arctic_speaker(speaker):
    print(f"Loading CMU ARCTIC speaker split: {speaker}")
    ds = load_dataset(CMU_DATASET_ID, split=speaker)
    rows = []
    for i, row in enumerate(tqdm(ds, desc=f"CMU {speaker}")):
        file_name = row.get("file", "")
        text = row.get("text", "")
        utt_id = normalize_prompt_text(text)
        if not utt_id:
          continue
        wav, sr = load_audio_any(row["audio"], TARGET_SR)
        path = AUDIO_WORK_DIR / "cmu" / speaker / f"{speaker}_{safe_id(utt_id)[:120]}.wav"
        if not path.exists():
            write_wav(path, wav, sr)
        rows.append({
            "source": "CMU_ARCTIC",
            "accent": "American",
            "speaker": speaker,
            "utt_id": utt_id,
            "text": text,
            "audio_path": str(path),
            "duration_sec": len(wav) / sr,
        })
    return rows

cmu_rows = []
for spk in CMU_NATIVE_SPEAKERS:
    cmu_rows.extend(load_cmu_arctic_speaker(spk))

cmu_df = pd.DataFrame(cmu_rows)
print(cmu_df.shape)
display(cmu_df.head())
print(cmu_df.groupby("speaker")["utt_id"].nunique())


## 4. Load L2-ARCTIC accented speech

This section supports three paths controlled by `L2_SOURCE`:

- `hf_gated`: loads `KoelLabs/L2Arctic` through `datasets.load_dataset(...)` after Hugging Face login.
- `hf_zip_mirror`: downloads speaker-level zip archives from `L2_ZIP_REPO` and extracts only selected speakers.
- `local_l2_root`: scans a manually downloaded/extracted L2-ARCTIC folder.

If the gated dataset schema changes, the flexible loader below tries to infer audio/speaker/text/file columns and writes each audio file into the notebook work directory.


In [ ]:
def selected_l2_speakers():
    pairs = []
    for accent, speakers in L2_SPEAKERS_BY_ACCENT.items():
        for spk in speakers[:SPEAKERS_PER_ACCENT]:
            pairs.append((accent, spk))
    return pairs

SELECTED_L2_SPEAKERS = selected_l2_speakers()
print("Selected L2 speakers:")
for accent, spk in SELECTED_L2_SPEAKERS:
    print(f"  {accent:10s} {spk}")


In [ ]:
def find_speaker_zip(repo_files, speaker):
    speaker_lower = speaker.lower()
    candidates = []
    for f in repo_files:
        base = Path(f).name.lower()
        if not base.endswith(".zip"):
            continue
        # Match ABA.zip, ABA_something.zip, speaker/ABA.zip, etc.
        if base == f"{speaker_lower}.zip" or base.startswith(f"{speaker_lower}_") or base.startswith(f"{speaker_lower}-"):
            candidates.append(f)
    if not candidates:
        # looser fallback
        for f in repo_files:
            if f.lower().endswith(".zip") and speaker_lower in Path(f).stem.lower().split("_"):
                candidates.append(f)
    if not candidates:
        raise FileNotFoundError(f"Could not find zip archive for speaker {speaker}. Inspect repo_files manually.")
    return sorted(candidates, key=len)[0]


def extract_zip_if_needed(zip_path, extract_root):
    zip_path = Path(zip_path)
    marker = extract_root / (zip_path.stem + ".extracted")
    if marker.exists():
        return
    extract_root.mkdir(exist_ok=True, parents=True)
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(extract_root)
    marker.touch()


def locate_speaker_dir(root, speaker):
    root = Path(root)
    candidates = []
    for p in root.rglob("*"):
        if p.is_dir() and p.name.upper() == speaker.upper():
            candidates.append(p)
    if not candidates:
        # Sometimes the zip itself extracts wav/transcript directly under a folder named by archive stem.
        for p in root.rglob("wav"):
            parent = p.parent
            if speaker.upper() in parent.name.upper():
                candidates.append(parent)
    if not candidates:
        raise FileNotFoundError(f"Could not locate extracted speaker directory for {speaker} under {root}")
    return sorted(candidates, key=lambda x: len(str(x)))[0]


def read_l2_transcripts(speaker_dir):
    transcript_dir_candidates = list(Path(speaker_dir).rglob("transcript")) + list(Path(speaker_dir).rglob("transcripts"))
    text_by_id = {}
    for tdir in transcript_dir_candidates:
        for txt in tdir.rglob("*.txt"):
            uid = extract_utt_id(txt.name)
            if uid is None:
                continue
            try:
                text = txt.read_text(encoding="utf-8", errors="ignore").strip()
            except Exception:
                text = ""
            text_by_id[uid] = normalize_ref_text(text)
    return text_by_id


def load_l2_from_speaker_dir(accent, speaker, speaker_dir):
    speaker_dir = Path(speaker_dir)
    wav_files = sorted([p for p in speaker_dir.rglob("*.wav") if "suitcase" not in str(p).lower()])
    text_by_id = read_l2_transcripts(speaker_dir)
    rows = []
    for wav_path in wav_files:
        uid = extract_utt_id(wav_path.name)
        if uid is None:
            continue
        text = text_by_id.get(uid, "")
        if not text:
            # Later we will fill text from CMU prompt if needed.
            text = ""
        rows.append({
            "source": "L2_ARCTIC",
            "accent": accent,
            "speaker": speaker,
            "utt_id": uid,
            "text": text,
            "audio_path": str(wav_path),
            "duration_sec": None,
        })
    return rows


def load_l2_from_zip_mirror():
    print("Listing L2 zip mirror files...")
    repo_files = list_repo_files(L2_ZIP_REPO, repo_type="dataset", token=HF_TOKEN)
    print("First 20 repo files:")
    for f in repo_files[:20]:
        print(" ", f)
    rows = []
    for accent, speaker in SELECTED_L2_SPEAKERS:
        zip_file = find_speaker_zip(repo_files, speaker)
        print(f"Downloading/finding {speaker}: {zip_file}")
        zip_path = hf_hub_download(
            repo_id=L2_ZIP_REPO,
            filename=zip_file,
            repo_type="dataset",
            cache_dir=str(CACHE_DIR / "hf"),
            token=HF_TOKEN,
        )
        extract_zip_if_needed(zip_path, L2_EXTRACT_DIR)
        spk_dir = locate_speaker_dir(L2_EXTRACT_DIR, speaker)
        spk_rows = load_l2_from_speaker_dir(accent, speaker, spk_dir)
        print(f"  {speaker}: {len(spk_rows)} wav rows from {spk_dir}")
        rows.extend(spk_rows)
    return pd.DataFrame(rows)


def load_l2_from_local_root():
    rows = []
    for accent, speaker in SELECTED_L2_SPEAKERS:
        spk_dir = locate_speaker_dir(LOCAL_L2_ROOT, speaker)
        rows.extend(load_l2_from_speaker_dir(accent, speaker, spk_dir))
    return pd.DataFrame(rows)


def normalize_prompt_text(text):
    import re
    text = str(text).lower().strip()
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def load_l2_from_gated_hf():
    """
    Loads KoelLabs/L2Arctic scripted split.

    This HF version does NOT expose an ARCTIC utterance ID column.
    So we use normalized transcript text as the shared prompt_id.
    """
    token_arg = HF_TOKEN if HF_TOKEN is not None else True

    #ds = load_dataset(
     #   L2_GATED_REPO,
      #  split=L2_GATED_SPLIT,
       # token=token_arg,
    #)

    ds = load_dataset(L2_GATED_REPO, split=L2_GATED_SPLIT, token=HF_TOKEN if HF_TOKEN is not None else True )

    print(ds)
    print(ds.column_names)

    required_cols = [
        "audio",
        "text",
        "speaker_code",
        "speaker_gender",
        "speaker_native_language",
    ]
    missing = [c for c in required_cols if c not in ds.column_names]
    if missing:
        raise ValueError(f"Missing required columns in L2 dataset: {missing}")

    rows = []

    for i, row in enumerate(tqdm(ds, desc="L2 gated HF")):
        speaker = row["speaker_code"]
        accent = row["speaker_native_language"]
        gender = row["speaker_gender"]
        text = row["text"]

        # Use transcript text as the common prompt key.
        prompt_id = normalize_prompt_text(text)

        wav, sr = load_audio_any(row["audio"], TARGET_SR)

        out_dir = AUDIO_WORK_DIR / "l2" / str(accent) / str(speaker)
        out_dir.mkdir(parents=True, exist_ok=True)

        path = out_dir / f"{speaker}_{i:05d}.wav"
        if not path.exists():
            sf.write(path, wav, sr)

        rows.append(
            {
                "dataset": "l2_arctic",
                "accent": accent,
                "speaker": speaker,
                "gender": gender,
                #"prompt_id": prompt_id,
                "prompt_id": normalize_prompt_text(text),
                "text": text,
                "path": str(path),
            }
        )

    return pd.DataFrame(rows)

if L2_SOURCE == "hf_zip_mirror":
    l2_df = load_l2_from_zip_mirror()
elif L2_SOURCE == "local_l2_root":
    l2_df = load_l2_from_local_root()
elif L2_SOURCE == "hf_gated":
    l2_df = load_l2_from_gated_hf()
else:
    raise ValueError(f"Unknown L2_SOURCE: {L2_SOURCE}")


# Normalize gated-HF schema to match the rest of the notebook
if "utt_id" not in l2_df.columns:
    l2_df["utt_id"] = l2_df["prompt_id"]

if "audio_path" not in l2_df.columns:
    l2_df["audio_path"] = l2_df["path"]

if "source" not in l2_df.columns:
    l2_df["source"] = l2_df.get("dataset", "L2_ARCTIC")

# Normalize accent labels
l2_df["accent"] = l2_df["accent"].replace({
    "Chinese": "Mandarin",
    "Mandarin Chinese": "Mandarin"
})

print(l2_df.shape)
display(l2_df.head())
print(l2_df.groupby(["accent", "speaker"])["utt_id"].nunique())


In [ ]:
print(sorted(l2_df["accent"].dropna().unique()))

## 6. Select the same 10 utterance IDs across all speakers/accent groups


In [ ]:
all_df = pd.concat([cmu_df, l2_df], ignore_index=True)
# Make display/schema clean after concatenating CMU and L2 rows
all_df["prompt_id"] = all_df["utt_id"]
all_df["path"] = all_df["audio_path"]


all_df["utt_id"] = all_df["utt_id"].astype(str).str.lower()
all_df["speaker"] = all_df["speaker"].astype(str)
all_df["accent"] = all_df["accent"].astype(str)

# Keep only selected speakers and required groups.
required_groups = ["American"] + list(L2_SPEAKERS_BY_ACCENT.keys())
print("Accent groups:", required_groups)

# Compute intersection over every speaker, not just every accent.
speaker_groups = all_df.groupby(["accent", "speaker"])["utt_id"].apply(set)
common_ids = None
for (accent, speaker), ids in speaker_groups.items():
    if accent not in required_groups:
        continue
    common_ids = set(ids) if common_ids is None else (common_ids & set(ids))

common_ids = sorted(common_ids)
print("Number of utterance IDs common to every selected speaker:", len(common_ids))
assert len(common_ids) >= N_PROMPTS, f"Only {len(common_ids)} common prompts found; reduce N_PROMPTS or inspect missing files."

# Use deterministic sample. You can also choose first N prompts by ID for reproducibility.
rng = random.Random(SEED)
selected_ids = sorted(rng.sample(common_ids, N_PROMPTS))
print("First 10 selected prompt IDs:", selected_ids[:10])

selected_df = all_df[all_df["utt_id"].isin(selected_ids)].copy()
selected_df = selected_df[selected_df["accent"].isin(required_groups)].copy()

# Make sure every accent has all selected IDs. Since American has multiple speakers, this checks speaker-level too.
coverage = selected_df.groupby(["accent", "speaker"])["utt_id"].nunique().reset_index(name="n_prompts")
display(coverage)
assert coverage["n_prompts"].min() == N_PROMPTS, "At least one selected speaker is missing selected prompts."

if MAX_ROWS_DEBUG is not None:
    selected_df = selected_df.sample(n=min(MAX_ROWS_DEBUG, len(selected_df)), random_state=SEED).copy()

selected_df = selected_df.sort_values(["utt_id", "accent", "speaker"]).reset_index(drop=True)
print("Selected rows:", selected_df.shape)
display(selected_df.head())

selected_df.to_csv(OUT_DIR / "selected_manifest.csv", index=False)


## 7. Listen to the first 5 matched prompts across accents

By default this plays one representative speaker per accent for each prompt. Set `LISTEN_ALL_SPEAKERS=True` if you want every speaker, but that will display many audio widgets.


In [ ]:
LISTEN_ALL_SPEAKERS = False
listen_ids = selected_ids[:5]

for uid in listen_ids:
    prompt_text = selected_df[selected_df["utt_id"] == uid]["text"].iloc[0]
    display(Markdown(f"### Prompt {uid}: {prompt_text}"))
    for accent in required_groups:
        sub = selected_df[(selected_df["utt_id"] == uid) & (selected_df["accent"] == accent)].copy()
        if not LISTEN_ALL_SPEAKERS:
            sub = sub.head(1)
        for _, row in sub.iterrows():
            display(Markdown(f"**{accent} / {row['speaker']}**"))
            display(IPAudio(row["audio_path"], rate=TARGET_SR))


## 8. Noise generation

For the first experiment, keep one controlled noise setting. Later you can sweep SNRs and noise types.


In [ ]:
def make_white_noise(n):
    return np.random.randn(n).astype(np.float32)


def make_pink_noise(n):
    # Simple Voss-ish approximation using filtered white noise.
    white = np.random.randn(n).astype(np.float32)
    b, a = butter(1, 0.02, btype="low")
    pink = lfilter(b, a, white).astype(np.float32)
    return pink


def make_babble_noise(n, pool_paths, exclude_path=None, k=5):
    candidates = [p for p in pool_paths if str(p) != str(exclude_path)]
    chosen = random.sample(candidates, k=min(k, len(candidates)))
    noise = np.zeros(n, dtype=np.float32)
    for p in chosen:
        wav, _ = load_audio_any(p, TARGET_SR)
        if len(wav) == 0:
            continue
        if len(wav) < n:
            reps = math.ceil(n / len(wav))
            wav = np.tile(wav, reps)
        start = random.randint(0, max(0, len(wav) - n))
        noise += wav[start:start+n]
    if rms(noise) < 1e-8:
        noise = make_white_noise(n)
    return noise.astype(np.float32)


def add_noise_at_snr(clean, noise, snr_db):
    clean = np.asarray(clean, dtype=np.float32)
    noise = np.asarray(noise, dtype=np.float32)
    if len(noise) < len(clean):
        noise = np.tile(noise, math.ceil(len(clean)/len(noise)))
    noise = noise[:len(clean)]
    clean_rms = rms(clean)
    noise_rms = rms(noise)
    target_noise_rms = clean_rms / (10 ** (snr_db / 20))
    scaled_noise = noise * (target_noise_rms / (noise_rms + 1e-12))
    noisy = clean + scaled_noise
    # Avoid clipping while preserving relative SNR approximately.
    peak = np.max(np.abs(noisy)) + 1e-12
    if peak > 0.99:
        noisy = noisy / peak * 0.99
    return noisy.astype(np.float32)


def make_noisy_version(audio_path, out_path, noise_type=NOISE_TYPE, snr_db=SNR_DB, pool_paths=None):
    clean, sr = load_audio_any(audio_path, TARGET_SR)
    n = len(clean)
    if noise_type == "white":
        noise = make_white_noise(n)
    elif noise_type == "pink":
        noise = make_pink_noise(n)
    elif noise_type == "babble":
        assert pool_paths is not None and len(pool_paths) > 0
        noise = make_babble_noise(n, pool_paths, exclude_path=audio_path)
    else:
        raise ValueError(noise_type)
    noisy = add_noise_at_snr(clean, noise, snr_db)
    write_wav(out_path, noisy, sr)
    return out_path


## 9. Speech enhancement model wrappers

The wrappers cache enhanced files so interrupted runs can resume.


In [ ]:
# Lazy model cache.
ENHANCER_MODELS = {}


def get_enhancer(name):
    if name in ENHANCER_MODELS:
        return ENHANCER_MODELS[name]
    if name == "sepformer_dns4":
        from speechbrain.inference.separation import SepformerSeparation
        model = SepformerSeparation.from_hparams(
            source="speechbrain/sepformer-dns4-16k-enhancement",
            savedir=str(CACHE_DIR / "speechbrain_sepformer_dns4"),
            run_opts={"device": TORCH_DEVICE},
        )
    elif name == "metricgan_plus":
        from speechbrain.inference.enhancement import SpectralMaskEnhancement
        model = SpectralMaskEnhancement.from_hparams(
            source="speechbrain/metricgan-plus-voicebank",
            savedir=str(CACHE_DIR / "speechbrain_metricgan_plus"),
            run_opts={"device": TORCH_DEVICE},
        )
    else:
        raise ValueError(f"Unknown enhancer: {name}")
    ENHANCER_MODELS[name] = model
    return model


def enhance_file(enhancer_name, in_path, out_path):
    out_path = Path(out_path)
    if out_path.exists():
        return out_path
    model = get_enhancer(enhancer_name)
    wav, sr = load_audio_any(in_path, TARGET_SR)
    wav_t = torch.tensor(wav, dtype=torch.float32, device=TORCH_DEVICE).unsqueeze(0)
    lengths = torch.tensor([1.0], device=TORCH_DEVICE)

    with torch.no_grad():
        if enhancer_name == "sepformer_dns4":
            enhanced = model.separate_batch(wav_t)
            # Expected shape sometimes [B, T, C] or [B, C, T]. Keep first stream robustly.
            if enhanced.ndim == 3:
                if enhanced.shape[1] == wav_t.shape[1]:
                    enhanced = enhanced[0, :, 0]
                else:
                    enhanced = enhanced[0, 0, :]
            else:
                enhanced = enhanced.squeeze()
        elif enhancer_name == "metricgan_plus":
            enhanced = model.enhance_batch(wav_t, lengths=lengths).squeeze()
        else:
            raise ValueError(enhancer_name)

    enhanced = enhanced.detach().cpu().numpy().astype(np.float32)
    if len(enhanced) == 0:
        enhanced = wav
    # Match original length for easier comparison.
    if len(enhanced) > len(wav):
        enhanced = enhanced[:len(wav)]
    elif len(enhanced) < len(wav):
        enhanced = np.pad(enhanced, (0, len(wav)-len(enhanced)))
    peak = np.max(np.abs(enhanced)) + 1e-12
    if peak > 1.0:
        enhanced = enhanced / peak * 0.99
    write_wav(out_path, enhanced, TARGET_SR)
    return out_path


## 10. Whisper ASR setup


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration

processor = WhisperProcessor.from_pretrained(WHISPER_MODEL)
whisper_model = WhisperForConditionalGeneration.from_pretrained(WHISPER_MODEL).to(TORCH_DEVICE)
whisper_model.eval()

# For openai/whisper-base.en, this is fine.
# For multilingual Whisper models, this forces English transcription.
try:
    forced_decoder_ids = processor.get_decoder_prompt_ids(language="english", task="transcribe")
    whisper_model.generation_config.forced_decoder_ids = forced_decoder_ids
except Exception:
    pass

def transcribe(path):
    wav, sr = load_audio_any(path, TARGET_SR)

    inputs = processor(
        wav,
        sampling_rate=TARGET_SR,
        return_tensors="pt",
    )

    input_features = inputs.input_features.to(TORCH_DEVICE)

    with torch.no_grad():
        predicted_ids = whisper_model.generate(input_features)

    text = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
    return text.strip()


## 11. Build audio conditions: raw, noisy, enhanced


In [ ]:
pool_paths = selected_df["audio_path"].tolist()
condition_rows = []

for _, row in tqdm(selected_df.iterrows(), total=len(selected_df), desc="Building audio conditions"):
    base_key = f"{safe_id(row['accent'])}_{safe_id(row['speaker'])}_{row['utt_id']}"
    raw_path = Path(row["audio_path"])
    noisy_path = AUDIO_WORK_DIR / "noisy" / f"{base_key}_{NOISE_TYPE}_{SNR_DB}dB.wav"
    if not noisy_path.exists():
        make_noisy_version(raw_path, noisy_path, NOISE_TYPE, SNR_DB, pool_paths=pool_paths)

    common_meta = {
        "accent": row["accent"],
        "speaker": row["speaker"],
        "utt_id": row["utt_id"],
        "reference": row["text"],
        "source": row["source"],
    }
    condition_rows.append({**common_meta, "condition": "raw", "audio_path": str(raw_path), "enhancer": "none"})
    condition_rows.append({**common_meta, "condition": "noisy", "audio_path": str(noisy_path), "enhancer": "none"})

    for enh in ENHANCERS_TO_RUN:
        enhanced_path = AUDIO_WORK_DIR / "enhanced" / enh / f"{base_key}_{NOISE_TYPE}_{SNR_DB}dB_{enh}.wav"
        enhance_file(enh, noisy_path, enhanced_path)
        condition_rows.append({**common_meta, "condition": enh, "audio_path": str(enhanced_path), "enhancer": enh})

conditions_df = pd.DataFrame(condition_rows)
print(conditions_df.shape)
display(conditions_df.head())
conditions_df.to_csv(OUT_DIR / "condition_manifest.csv", index=False)


## 12. Run Whisper ASR

This is the slowest cell. It writes partial results after every item, so you can stop and resume.


In [ ]:
RESULTS_PATH = OUT_DIR / "asr_results_partial.csv"

if RESULTS_PATH.exists():
    results_df = pd.read_csv(RESULTS_PATH)
    done_keys = set(results_df["result_key"].astype(str))
    results = results_df.to_dict("records")
    print(f"Resuming from {len(done_keys)} completed ASR rows.")
else:
    done_keys = set()
    results = []

for _, row in tqdm(conditions_df.iterrows(), total=len(conditions_df), desc="Whisper ASR"):
    result_key = f"{row['accent']}|{row['speaker']}|{row['utt_id']}|{row['condition']}"
    if result_key in done_keys:
        continue
    hyp = transcribe(row["audio_path"])
    ref = str(row["reference"])
    try:
        utt_wer = wer(text_norm(ref), text_norm(hyp))
        utt_cer = cer(text_norm(ref), text_norm(hyp))
    except Exception:
        utt_wer = np.nan
        utt_cer = np.nan
    rec = {
        **row.to_dict(),
        "hypothesis": hyp,
        "utt_wer": utt_wer,
        "utt_cer": utt_cer,
        "result_key": result_key,
    }
    results.append(rec)
    done_keys.add(result_key)
    pd.DataFrame(results).to_csv(RESULTS_PATH, index=False)

results_df = pd.DataFrame(results)
results_df.to_csv(OUT_DIR / "asr_results.csv", index=False)
print(results_df.shape)
display(results_df.head())


## 13. Aggregate WER/CER correctly by accent and condition

Do not rely only on mean utterance WER. The standard corpus-level WER is computed after concatenating references/hypotheses within a group.


In [ ]:
def group_asr_metrics(df):
    rows = []
    group_cols = ["accent", "condition"]
    for keys, g in df.groupby(group_cols):
        refs = [text_norm(str(x)) for x in g["reference"].tolist()]
        hyps = [text_norm(str(x)) for x in g["hypothesis"].tolist()]
        rows.append({
            "accent": keys[0],
            "condition": keys[1],
            "n": len(g),
            "corpus_wer": wer(refs, hyps),
            "corpus_cer": cer(refs, hyps),
            "mean_utt_wer": float(np.nanmean(g["utt_wer"])),
            "mean_utt_cer": float(np.nanmean(g["utt_cer"])),
        })
    return pd.DataFrame(rows)

summary = group_asr_metrics(results_df)
summary = summary.sort_values(["accent", "condition"]).reset_index(drop=True)
display(summary)
summary.to_csv(OUT_DIR / "accent_condition_summary.csv", index=False)


## 14. Compute absolute and relative degradation

We compute degradation relative to both `raw` and `noisy`, but for enhancement models the most important comparison is usually:

\[
\Delta WER_{vs\ noisy} = WER(Enhanced(Noisy)) - WER(Noisy)
\]

Positive means the enhancement frontend made Whisper worse than leaving the noisy audio alone.


In [ ]:
piv = summary.pivot(index="accent", columns="condition", values="corpus_wer")
for cond in summary["condition"].unique():
    if cond in ["raw", "noisy"]:
        continue
    piv[f"{cond}_delta_vs_raw"] = piv[cond] - piv["raw"]
    piv[f"{cond}_rel_vs_raw"] = (piv[cond] - piv["raw"]) / (piv["raw"] + 1e-12)
    piv[f"{cond}_delta_vs_noisy"] = piv[cond] - piv["noisy"]
    piv[f"{cond}_rel_vs_noisy"] = (piv[cond] - piv["noisy"]) / (piv["noisy"] + 1e-12)

piv = piv.reset_index()
display(piv)
piv.to_csv(OUT_DIR / "degradation_pivot.csv", index=False)


## 15. Utterance-level helped/hurt analysis


In [ ]:
utt_piv = results_df.pivot_table(
    index=["accent", "speaker", "utt_id", "reference"],
    columns="condition",
    values="utt_wer",
    aggfunc="first"
).reset_index()

for enh in ENHANCERS_TO_RUN:
    if enh in utt_piv.columns:
        utt_piv[f"{enh}_utt_delta_vs_noisy"] = utt_piv[enh] - utt_piv["noisy"]
        utt_piv[f"{enh}_hurt_vs_noisy"] = utt_piv[f"{enh}_utt_delta_vs_noisy"] > 0
        utt_piv[f"{enh}_helped_vs_noisy"] = utt_piv[f"{enh}_utt_delta_vs_noisy"] < 0

hurt_rows = []
for enh in ENHANCERS_TO_RUN:
    if f"{enh}_hurt_vs_noisy" not in utt_piv.columns:
        continue
    for accent, g in utt_piv.groupby("accent"):
        hurt_rows.append({
            "accent": accent,
            "enhancer": enh,
            "n": len(g),
            "mean_delta_utt_wer_vs_noisy": float(g[f"{enh}_utt_delta_vs_noisy"].mean()),
            "frac_hurt_vs_noisy": float(g[f"{enh}_hurt_vs_noisy"].mean()),
            "frac_helped_vs_noisy": float(g[f"{enh}_helped_vs_noisy"].mean()),
        })

hurt_summary = pd.DataFrame(hurt_rows)
display(hurt_summary)
utt_piv.to_csv(OUT_DIR / "utterance_level_pivot.csv", index=False)
hurt_summary.to_csv(OUT_DIR / "hurt_helped_summary.csv", index=False)


## 16. Bootstrap confidence intervals by accent/condition


In [ ]:
def bootstrap_wer_ci(g, n_boot=300, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = g.reset_index(drop=True)
    vals = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(rows), size=len(rows))
        refs = [text_norm(str(x)) for x in rows.loc[idx, "reference"].tolist()]
        hyps = [text_norm(str(x)) for x in rows.loc[idx, "hypothesis"].tolist()]
        vals.append(wer(refs, hyps))
    return np.percentile(vals, [2.5, 50, 97.5])

ci_rows = []
for (accent, condition), g in tqdm(results_df.groupby(["accent", "condition"]), desc="Bootstrap CIs"):
    lo, med, hi = bootstrap_wer_ci(g, n_boot=300)
    ci_rows.append({"accent": accent, "condition": condition, "wer_ci_low": lo, "wer_boot_median": med, "wer_ci_high": hi})

ci_df = pd.DataFrame(ci_rows)
summary_ci = summary.merge(ci_df, on=["accent", "condition"], how="left")
display(summary_ci)
summary_ci.to_csv(OUT_DIR / "summary_with_bootstrap_ci.csv", index=False)


## 17. Plots


In [ ]:
def savefig(name):
    path = OUT_DIR / name
    plt.tight_layout()
    plt.savefig(path, dpi=200, bbox_inches="tight")
    print("Saved", path)
    plt.show()

# 1. WER by accent and condition.
plot_df = summary.copy()
conditions_order = ["raw", "noisy"] + [e for e in ENHANCERS_TO_RUN if e in plot_df["condition"].unique()]
accents_order = required_groups

x = np.arange(len(accents_order))
width = 0.8 / len(conditions_order)
plt.figure(figsize=(13, 5))
for i, cond in enumerate(conditions_order):
    vals = []
    for acc in accents_order:
        v = plot_df[(plot_df["accent"] == acc) & (plot_df["condition"] == cond)]["corpus_wer"]
        vals.append(float(v.iloc[0]) if len(v) else np.nan)
    plt.bar(x + i*width - 0.4 + width/2, vals, width, label=cond)
plt.xticks(x, accents_order, rotation=30, ha="right")
plt.ylabel("Corpus WER")
plt.title("Whisper WER by accent and audio condition")
plt.legend()
savefig("wer_by_accent_condition.png")

# 2. Delta WER vs noisy for enhancement models.
plt.figure(figsize=(12, 5))
width = 0.8 / max(1, len(ENHANCERS_TO_RUN))
for i, enh in enumerate(ENHANCERS_TO_RUN):
    col = f"{enh}_delta_vs_noisy"
    if col not in piv.columns:
        continue
    vals = [float(piv[piv["accent"] == acc][col].iloc[0]) for acc in accents_order]
    plt.bar(x + i*width - 0.4 + width/2, vals, width, label=enh)
plt.axhline(0, linewidth=1)
plt.xticks(x, accents_order, rotation=30, ha="right")
plt.ylabel("Δ WER vs noisy")
plt.title("Enhancement degradation/improvement relative to noisy input")
plt.legend()
savefig("delta_wer_vs_noisy.png")

# 3. Relative delta WER vs noisy.
plt.figure(figsize=(12, 5))
for i, enh in enumerate(ENHANCERS_TO_RUN):
    col = f"{enh}_rel_vs_noisy"
    if col not in piv.columns:
        continue
    vals = [100*float(piv[piv["accent"] == acc][col].iloc[0]) for acc in accents_order]
    plt.bar(x + i*width - 0.4 + width/2, vals, width, label=enh)
plt.axhline(0, linewidth=1)
plt.xticks(x, accents_order, rotation=30, ha="right")
plt.ylabel("Relative Δ WER vs noisy (%)")
plt.title("Relative ASR degradation after enhancement")
plt.legend()
savefig("relative_delta_wer_vs_noisy.png")

# 4. Fraction hurt vs noisy.
if len(hurt_summary):
    plt.figure(figsize=(12, 5))
    for i, enh in enumerate(ENHANCERS_TO_RUN):
        sub = hurt_summary[hurt_summary["enhancer"] == enh]
        vals = []
        for acc in accents_order:
            v = sub[sub["accent"] == acc]["frac_hurt_vs_noisy"]
            vals.append(float(v.iloc[0]) if len(v) else np.nan)
        plt.bar(x + i*width - 0.4 + width/2, vals, width, label=enh)
    plt.xticks(x, accents_order, rotation=30, ha="right")
    plt.ylabel("Fraction of utterances hurt")
    plt.title("How often enhancement increases utterance WER vs noisy")
    plt.legend()
    savefig("fraction_hurt_vs_noisy.png")


## 18. Optional: simple signal-level diagnostics

These are not the main claim, but they help catch confounds such as duration changes, clipping, or extreme loudness shifts.


In [ ]:
def signal_diagnostics(path):
    wav, sr = load_audio_any(path, TARGET_SR)
    peak = float(np.max(np.abs(wav)) + 1e-12)
    return {
        "duration_sec": len(wav)/sr,
        "rms": rms(wav),
        "peak": peak,
        "clipped_frac": float(np.mean(np.abs(wav) >= 0.999)),
    }

# Run diagnostics on all condition files. This is usually fast.
diag_rows = []
for _, row in tqdm(conditions_df.iterrows(), total=len(conditions_df), desc="Signal diagnostics"):
    d = signal_diagnostics(row["audio_path"])
    diag_rows.append({
        "accent": row["accent"],
        "speaker": row["speaker"],
        "utt_id": row["utt_id"],
        "condition": row["condition"],
        **d,
    })

diag_df = pd.DataFrame(diag_rows)
display(diag_df.groupby(["accent", "condition"])[["duration_sec", "rms", "peak", "clipped_frac"]].mean().reset_index())
diag_df.to_csv(OUT_DIR / "signal_diagnostics.csv", index=False)


## 19. Final interpretation helper

This cell prints the strongest candidate findings to inspect.


In [ ]:
print("=== Corpus WER summary ===")
display(summary.sort_values(["accent", "condition"]))

print("=== Degradation pivot ===")
display(piv)

for enh in ENHANCERS_TO_RUN:
    col = f"{enh}_delta_vs_noisy"
    if col in piv.columns:
        print(f"\nLargest WER increases vs noisy for {enh}:")
        display(piv[["accent", "raw", "noisy", enh, col, f"{enh}_rel_vs_noisy"]].sort_values(col, ascending=False))

print("Files saved in:", OUT_DIR.resolve())


In this small matched-prompt smoke test, speech enhancement did not improve Whisper ASR uniformly across accents. Both enhancement models substantially improved American and Korean noisy speech, but degraded several other accented groups. The degradation pattern was model-specific: SepFormer strongly degraded Mandarin and Spanish, while MetricGAN+ strongly degraded Hindi.

Raw WER: accented groups are worse than American, as expected.
Noisy WER: degradation is not uniform.
Enhanced WER: enhancement sometimes helps, sometimes catastrophically hurts.
Model-specific failure: SepFormer and MetricGAN+ hurt different accent groups.

## 15. Things this notebook checks beyond the basic pipeline

- **Prompt congruence:** it computes the intersection of utterance IDs across every selected speaker, not just across accent labels.
- **Reference consistency:** it uses CMU ARCTIC prompt text as the canonical transcript wherever possible.
- **Controlled noise:** it applies the same synthetic noise recipe across groups.
- **Multiple comparison levels:** it reports accent-level and speaker-level degradation.
- **Bootstrap CIs:** it estimates confidence intervals for mean WER changes by accent/model.
- **Helped vs hurt rate:** it tracks how often enhancement improves or degrades utterances, not only the mean.

Recommended first run: keep `SPEAKERS_PER_ACCENT = 1`, `N_PROMPTS = 10`, and one enhancer. Once that works, switch to `SPEAKERS_PER_ACCENT = 4`, `N_PROMPTS = 100`.


In [ ]:
# ============================================================
# A. Noise-only degradation: raw -> noisy
# ============================================================

noise_piv = results_df.pivot_table(
    index=["accent", "speaker", "utt_id", "reference"],
    columns="condition",
    values="utt_wer",
    aggfunc="first"
).reset_index()

noise_piv["noise_delta_wer"] = noise_piv["noisy"] - noise_piv["raw"]
noise_piv["noise_rel_delta_wer"] = noise_piv["noise_delta_wer"] / (noise_piv["raw"] + 1e-12)
noise_piv["noise_hurts"] = noise_piv["noise_delta_wer"] > 0

noise_summary = (
    noise_piv
    .groupby("accent")
    .agg(
        n=("utt_id", "count"),
        raw_mean_utt_wer=("raw", "mean"),
        noisy_mean_utt_wer=("noisy", "mean"),
        mean_noise_delta_wer=("noise_delta_wer", "mean"),
        median_noise_delta_wer=("noise_delta_wer", "median"),
        frac_noise_hurts=("noise_hurts", "mean"),
    )
    .reset_index()
    .sort_values("mean_noise_delta_wer", ascending=False)
)

display(noise_summary)
noise_summary.to_csv(OUT_DIR / "noise_only_degradation_summary.csv", index=False)
noise_piv.to_csv(OUT_DIR / "noise_only_utterance_pivot.csv", index=False)

In [ ]:
# ============================================================
# B. Plot raw vs noisy WER by accent
# ============================================================

plt.figure(figsize=(10, 5))

x = np.arange(len(noise_summary))
width = 0.35

plt.bar(x - width/2, noise_summary["raw_mean_utt_wer"], width, label="raw")
plt.bar(x + width/2, noise_summary["noisy_mean_utt_wer"], width, label="noisy")

plt.xticks(x, noise_summary["accent"], rotation=30, ha="right")
plt.ylabel("Mean utterance WER")
plt.title("Noise-only ASR degradation: raw vs noisy")
plt.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / "noise_raw_vs_noisy_by_accent.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# C. Plot noise-induced ΔWER by accent
# ============================================================

plt.figure(figsize=(10, 5))

plot_df = noise_summary.sort_values("mean_noise_delta_wer", ascending=False)

plt.bar(plot_df["accent"], plot_df["mean_noise_delta_wer"])
plt.axhline(0, linewidth=1)

plt.xticks(rotation=30, ha="right")
plt.ylabel("Mean Δ utterance WER: noisy - raw")
plt.title("How much does controlled noise degrade Whisper ASR?")
plt.tight_layout()
plt.savefig(OUT_DIR / "noise_delta_wer_by_accent.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# ============================================================
# D. Relationship between noise degradation and enhancement effect
# ============================================================

enh_effect_rows = []

for enh in ENHANCERS_TO_RUN:
    if enh not in noise_piv.columns:
        continue

    tmp = noise_piv.copy()
    tmp[f"{enh}_delta_vs_noisy"] = tmp[enh] - tmp["noisy"]

    grouped = (
        tmp.groupby("accent")
        .agg(
            mean_noise_delta_wer=("noise_delta_wer", "mean"),
            mean_enh_delta_vs_noisy=(f"{enh}_delta_vs_noisy", "mean"),
        )
        .reset_index()
    )
    grouped["enhancer"] = enh
    enh_effect_rows.append(grouped)

noise_vs_enh = pd.concat(enh_effect_rows, ignore_index=True)
display(noise_vs_enh)

noise_vs_enh.to_csv(OUT_DIR / "noise_vs_enhancement_effect.csv", index=False)

for enh in ENHANCERS_TO_RUN:
    sub = noise_vs_enh[noise_vs_enh["enhancer"] == enh].copy()

    plt.figure(figsize=(6, 5))
    plt.scatter(sub["mean_noise_delta_wer"], sub["mean_enh_delta_vs_noisy"])

    for _, row in sub.iterrows():
        plt.annotate(
            row["accent"],
            (row["mean_noise_delta_wer"], row["mean_enh_delta_vs_noisy"]),
            textcoords="offset points",
            xytext=(5, 5),
            fontsize=9,
        )

    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)
    plt.xlabel("Mean noise degradation: noisy - raw")
    plt.ylabel(f"Mean enhancement effect: {enh} - noisy")
    plt.title(f"Noise degradation vs enhancement effect: {enh}")
    plt.tight_layout()
    plt.savefig(OUT_DIR / f"noise_vs_enhancement_{enh}.png", dpi=200, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# E. Worst noise-only failures
# ============================================================

worst_noise = (
    noise_piv
    .sort_values("noise_delta_wer", ascending=False)
    .reset_index(drop=True)
)

display(worst_noise[
    ["accent", "speaker", "utt_id", "reference", "raw", "noisy", "noise_delta_wer"]
].head(20))

worst_noise.to_csv(OUT_DIR / "worst_noise_failures.csv", index=False)

In [ ]:
# ============================================================
# F. Listen to raw, noisy, and enhanced audio for selected failures
# ============================================================

def get_condition_audio_path(accent, speaker, utt_id, condition):
    sub = conditions_df[
        (conditions_df["accent"].astype(str) == str(accent)) &
        (conditions_df["speaker"].astype(str) == str(speaker)) &
        (conditions_df["utt_id"].astype(str) == str(utt_id)) &
        (conditions_df["condition"].astype(str) == str(condition))
    ]

    if len(sub) == 0:
        return None

    return sub.iloc[0]["audio_path"]


def get_condition_hypothesis(accent, speaker, utt_id, condition):
    sub = results_df[
        (results_df["accent"].astype(str) == str(accent)) &
        (results_df["speaker"].astype(str) == str(speaker)) &
        (results_df["utt_id"].astype(str) == str(utt_id)) &
        (results_df["condition"].astype(str) == str(condition))
    ]

    if len(sub) == 0:
        return None

    return sub.iloc[0]["hypothesis"]


def listen_case(row, conditions=("raw", "noisy", "sepformer_dns4", "metricgan_plus")):
    accent = row["accent"]
    speaker = row["speaker"]
    utt_id = row["utt_id"]
    ref = row["reference"]

    display(Markdown(f"## {accent} / {speaker}"))
    display(Markdown(f"**Prompt ID:** `{utt_id}`"))
    display(Markdown(f"**Reference:** {ref}"))

    for cond in conditions:
        path = get_condition_audio_path(accent, speaker, utt_id, cond)
        hyp = get_condition_hypothesis(accent, speaker, utt_id, cond)

        if path is None:
            display(Markdown(f"### {cond}: missing audio"))
            continue

        wer_val = results_df[
            (results_df["accent"].astype(str) == str(accent)) &
            (results_df["speaker"].astype(str) == str(speaker)) &
            (results_df["utt_id"].astype(str) == str(utt_id)) &
            (results_df["condition"].astype(str) == str(cond))
        ]["utt_wer"]

        wer_text = float(wer_val.iloc[0]) if len(wer_val) else np.nan

        display(Markdown(f"### {cond} | WER={wer_text:.3f}"))
        display(IPAudio(path, rate=TARGET_SR))

        if hyp is not None:
            display(Markdown(f"**Whisper:** {hyp}"))


# Listen to top K noise failures
TOP_K_LISTEN = 5

for i in range(min(TOP_K_LISTEN, len(worst_noise))):
    listen_case(worst_noise.iloc[i])

In [ ]:
# ============================================================
# G. Listen to worst enhancement failures by model
# ============================================================

for enh in ENHANCERS_TO_RUN:
    if enh not in noise_piv.columns:
        continue

    tmp = noise_piv.copy()
    tmp[f"{enh}_delta_vs_noisy"] = tmp[enh] - tmp["noisy"]
    worst_enh = tmp.sort_values(f"{enh}_delta_vs_noisy", ascending=False).reset_index(drop=True)

    display(Markdown(f"# Worst enhancement failures: {enh}"))
    display(worst_enh[
        ["accent", "speaker", "utt_id", "reference", "raw", "noisy", enh, f"{enh}_delta_vs_noisy"]
    ].head(10))

    for i in range(min(3, len(worst_enh))):
        listen_case(worst_enh.iloc[i], conditions=("raw", "noisy", enh))

In [ ]:
# ============================================================
# H. Listen to worst noise and enhancement failures per accent
# ============================================================

def listen_worst_per_accent(enhancer=None, top_per_accent=1):
    if enhancer is None:
        metric_col = "noise_delta_wer"
        title = "Worst noise-only failures per accent"
        df = noise_piv.copy()
        conditions = ("raw", "noisy")
    else:
        metric_col = f"{enhancer}_delta_vs_noisy"
        title = f"Worst enhancement failures per accent: {enhancer}"
        df = noise_piv.copy()
        df[metric_col] = df[enhancer] - df["noisy"]
        conditions = ("raw", "noisy", enhancer)

    display(Markdown(f"# {title}"))

    examples = (
        df.sort_values(metric_col, ascending=False)
        .groupby("accent")
        .head(top_per_accent)
        .sort_values(["accent", metric_col], ascending=[True, False])
    )

    display(examples[["accent", "speaker", "utt_id", "reference", metric_col]])

    for _, row in examples.iterrows():
        listen_case(row, conditions=conditions)


# Noise-only examples
listen_worst_per_accent(enhancer=None, top_per_accent=1)

# Enhancement examples
for enh in ENHANCERS_TO_RUN:
    listen_worst_per_accent(enhancer=enh, top_per_accent=1)

In [ ]:
# ============================================================
# I. Signal diagnostics for worst failures
# ============================================================

def signal_diagnostics_for_case(accent, speaker, utt_id, conditions=("raw", "noisy", "sepformer_dns4", "metricgan_plus")):
    rows = []

    for cond in conditions:
        path = get_condition_audio_path(accent, speaker, utt_id, cond)
        if path is None:
            continue

        wav, sr = load_audio_any(path, TARGET_SR)

        rows.append({
            "accent": accent,
            "speaker": speaker,
            "utt_id": utt_id,
            "condition": cond,
            "duration_sec": len(wav) / sr,
            "rms": rms(wav),
            "peak": float(np.max(np.abs(wav)) + 1e-12),
            "clipped_frac": float(np.mean(np.abs(wav) >= 0.999)),
            "path": path,
        })

    return pd.DataFrame(rows)


diag_failure_rows = []

# Top noise failures
for _, row in worst_noise.head(10).iterrows():
    d = signal_diagnostics_for_case(row["accent"], row["speaker"], row["utt_id"])
    d["failure_type"] = "noise_only"
    diag_failure_rows.append(d)

# Top enhancement failures
for enh in ENHANCERS_TO_RUN:
    tmp = noise_piv.copy()
    tmp[f"{enh}_delta_vs_noisy"] = tmp[enh] - tmp["noisy"]
    worst_enh = tmp.sort_values(f"{enh}_delta_vs_noisy", ascending=False).head(10)

    for _, row in worst_enh.iterrows():
        d = signal_diagnostics_for_case(row["accent"], row["speaker"], row["utt_id"])
        d["failure_type"] = f"{enh}_failure"
        diag_failure_rows.append(d)

failure_diag_df = pd.concat(diag_failure_rows, ignore_index=True)
display(failure_diag_df)

failure_diag_df.to_csv(OUT_DIR / "failure_case_signal_diagnostics.csv", index=False)

In [ ]:
# ============================================================
# J. Quick interpretation helper for noise-related degradation
# ============================================================

print("=== Noise-only degradation by accent ===")
display(noise_summary)

print("\n=== Most noise-sensitive accents ===")
display(noise_summary.sort_values("mean_noise_delta_wer", ascending=False).head(3))

print("\n=== Least noise-sensitive accents ===")
display(noise_summary.sort_values("mean_noise_delta_wer", ascending=True).head(3))

for enh in ENHANCERS_TO_RUN:
    col = f"{enh}_delta_vs_noisy"
    if col not in noise_piv.columns:
        continue

    tmp = noise_piv.copy()
    tmp[col] = tmp[enh] - tmp["noisy"]

    enh_summary = (
        tmp.groupby("accent")
        .agg(
            mean_delta_vs_noisy=(col, "mean"),
            frac_hurt_vs_noisy=(col, lambda x: float(np.mean(x > 0))),
        )
        .reset_index()
        .sort_values("mean_delta_vs_noisy", ascending=False)
    )

    print(f"\n=== Enhancement degradation summary: {enh} ===")
    display(enh_summary)

In [ ]:
# ============================================================
# Export all key outputs into a single PDF report
# ============================================================

%pip install -q reportlab pillow

from pathlib import Path
import textwrap
import pandas as pd
from PIL import Image

from reportlab.lib import colors
from reportlab.lib.pagesizes import letter, landscape
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, PageBreak,
    Table, TableStyle, Image as RLImage
)

REPORT_PATH = OUT_DIR / f"accent_enhancement_audit_report_{NOISE_TYPE}_{SNR_DB}dB.pdf"

styles = getSampleStyleSheet()
styles.add(ParagraphStyle(
    name="Small",
    parent=styles["BodyText"],
    fontSize=8,
    leading=10,
))
styles.add(ParagraphStyle(
    name="Tiny",
    parent=styles["BodyText"],
    fontSize=6.5,
    leading=8,
))
styles.add(ParagraphStyle(
    name="Section",
    parent=styles["Heading2"],
    spaceBefore=12,
    spaceAfter=8,
))


def fmt(x, ndigits=3):
    if pd.isna(x):
        return ""
    if isinstance(x, float):
        return f"{x:.{ndigits}f}"
    return str(x)


def add_df_table(story, df, title, max_rows=25, max_col_width_chars=34):
    story.append(Paragraph(title, styles["Section"]))

    if df is None or len(df) == 0:
        story.append(Paragraph("No data available.", styles["BodyText"]))
        story.append(Spacer(1, 0.15 * inch))
        return

    show_df = df.copy().head(max_rows)

    # Convert all values to short strings.
    table_data = [list(show_df.columns)]
    for _, row in show_df.iterrows():
        table_data.append([
            textwrap.shorten(fmt(v), width=max_col_width_chars, placeholder="...")
            for v in row.tolist()
        ])

    table = Table(table_data, repeatRows=1)

    table.setStyle(TableStyle([
        ("BACKGROUND", (0, 0), (-1, 0), colors.HexColor("#E8E8E8")),
        ("TEXTCOLOR", (0, 0), (-1, 0), colors.black),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("FONTNAME", (0, 1), (-1, -1), "Helvetica"),
        ("FONTSIZE", (0, 0), (-1, -1), 6.2),
        ("LEADING", (0, 0), (-1, -1), 7),
        ("GRID", (0, 0), (-1, -1), 0.25, colors.HexColor("#BBBBBB")),
        ("VALIGN", (0, 0), (-1, -1), "TOP"),
        ("ROWBACKGROUNDS", (0, 1), (-1, -1), [colors.white, colors.HexColor("#F7F7F7")]),
    ]))

    story.append(table)

    if len(df) > max_rows:
        story.append(Spacer(1, 0.08 * inch))
        story.append(Paragraph(f"Showing first {max_rows} of {len(df)} rows.", styles["Small"]))

    story.append(Spacer(1, 0.2 * inch))


def add_png(story, png_path, title, max_width=9.2 * inch, max_height=6.2 * inch):
    png_path = Path(png_path)
    if not png_path.exists():
        return

    story.append(Paragraph(title, styles["Section"]))

    with Image.open(png_path) as im:
        w, h = im.size

    scale = min(max_width / w, max_height / h)
    img = RLImage(str(png_path), width=w * scale, height=h * scale)
    story.append(img)
    story.append(Spacer(1, 0.2 * inch))


def safe_read_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    return None


# Load outputs saved by your notebook.
summary_csv = safe_read_csv(OUT_DIR / "accent_condition_summary.csv")
degradation_csv = safe_read_csv(OUT_DIR / "degradation_pivot.csv")
hurt_csv = safe_read_csv(OUT_DIR / "hurt_helped_summary.csv")
noise_summary_csv = safe_read_csv(OUT_DIR / "noise_only_degradation_summary.csv")
noise_vs_enh_csv = safe_read_csv(OUT_DIR / "noise_vs_enhancement_effect.csv")
worst_noise_csv = safe_read_csv(OUT_DIR / "worst_noise_failures.csv")
signal_diag_csv = safe_read_csv(OUT_DIR / "signal_diagnostics.csv")
failure_diag_csv = safe_read_csv(OUT_DIR / "failure_case_signal_diagnostics.csv")


doc = SimpleDocTemplate(
    str(REPORT_PATH),
    pagesize=landscape(letter),
    rightMargin=0.35 * inch,
    leftMargin=0.35 * inch,
    topMargin=0.35 * inch,
    bottomMargin=0.35 * inch,
)

story = []

story.append(Paragraph("Accent-dependent ASR Degradation After Speech Enhancement", styles["Title"]))
story.append(Spacer(1, 0.15 * inch))

actual_speakers_per_accent = (
    results_df[results_df["condition"] == "raw"]
    .groupby("accent")["speaker"]
    .nunique()
    .to_dict()
)

actual_prompts_per_accent = (
    results_df[results_df["condition"] == "raw"]
    .groupby("accent")["utt_id"]
    .nunique()
    .to_dict()
)

metadata_text = f"""
<b>Noise type:</b> {NOISE_TYPE}<br/>
<b>SNR:</b> {SNR_DB} dB<br/>
<b>Whisper model:</b> {WHISPER_MODEL}<br/>
<b>Enhancers:</b> {", ".join(ENHANCERS_TO_RUN)}<br/>
<b>Actual speakers per accent:</b> {actual_speakers_per_accent}<br/>
<b>Actual prompts per accent:</b> {actual_prompts_per_accent}<br/>
<b>Output directory:</b> {OUT_DIR.resolve()}
"""
story.append(Paragraph(metadata_text, styles["BodyText"]))
interpretation = f"""
This report summarizes a matched-prompt smoke test for accent-dependent ASR degradation after controlled noise addition and speech enhancement.
The key comparison is enhanced WER minus noisy WER. Positive values mean enhancement made Whisper worse than leaving the noisy audio unchanged.
The noise-only section checks whether the controlled noise condition itself affects accent groups unevenly before enhancement.
"""
story.append(Paragraph(interpretation, styles["BodyText"]))

story.append(PageBreak())

# Main ASR outputs
add_df_table(story, summary_csv, "Corpus WER/CER by Accent and Condition", max_rows=40)
add_df_table(story, degradation_csv, "Degradation Pivot: Enhanced vs Raw and Noisy", max_rows=20)
add_df_table(story, hurt_csv, "Utterance-level Helped/Hurt Summary", max_rows=30)

story.append(PageBreak())

# Noise diagnostics
add_df_table(story, noise_summary_csv, "Noise-only Degradation Summary: Raw -> Noisy", max_rows=20)
add_df_table(story, noise_vs_enh_csv, "Noise Degradation vs Enhancement Effect", max_rows=30)
add_df_table(story, worst_noise_csv, "Worst Noise-only Failures", max_rows=20)

story.append(PageBreak())

# Signal diagnostics
if signal_diag_csv is not None:
    signal_grouped = (
        signal_diag_csv
        .groupby(["accent", "condition"])[["duration_sec", "rms", "peak", "clipped_frac"]]
        .mean()
        .reset_index()
    )
else:
    signal_grouped = None

add_df_table(story, signal_grouped, "Mean Signal Diagnostics by Accent and Condition", max_rows=60)
add_df_table(story, failure_diag_csv, "Signal Diagnostics for Worst Failure Cases", max_rows=40)

story.append(PageBreak())

# Plots from original notebook and added noise cells
plot_files = [
    ("WER by Accent and Condition", OUT_DIR / "wer_by_accent_condition.png"),
    ("Delta WER vs Noisy", OUT_DIR / "delta_wer_vs_noisy.png"),
    ("Relative Delta WER vs Noisy", OUT_DIR / "relative_delta_wer_vs_noisy.png"),
    ("Fraction Hurt vs Noisy", OUT_DIR / "fraction_hurt_vs_noisy.png"),
    ("Noise Raw vs Noisy by Accent", OUT_DIR / "noise_raw_vs_noisy_by_accent.png"),
    ("Noise Delta WER by Accent", OUT_DIR / "noise_delta_wer_by_accent.png"),
]

# Add enhancer-specific noise-vs-enhancement plots if they exist.
for enh in ENHANCERS_TO_RUN:
    plot_files.append((
        f"Noise Degradation vs Enhancement Effect: {enh}",
        OUT_DIR / f"noise_vs_enhancement_{enh}.png"
    ))

for title, path in plot_files:
    if Path(path).exists():
        add_png(story, path, title)

# Build PDF
doc.build(story)

print(f"Saved PDF report to: {REPORT_PATH}")

## 20. Things to add after the first run


1. **Noise sweep**: 0, 5, 10, 20 dB SNR.
2. **Noise type sweep**: babble, cafe, street, music, white/pink noise.
3. **Whisper robustness sweep**: `base.en`, `small.en`, `medium.en`, maybe `large-v3`.
4. **More enhancement models**: DeepFilterNet and Facebook/Meta Denoiser are good deployment-style additions.
5. **Phoneme-level error analysis**: use L2-ARCTIC annotations to check whether enhancement changes errors around accent-sensitive phones.
6. **Matched-prompt mixed-effects model**: model WER with accent/enhancer fixed effects and speaker/prompt random effects.
7. **Audio metric mismatch**: add PESQ/STOI/DNSMOS if setup is stable, then test whether perceptual improvement predicts ASR improvement equally across accents.
